Environment: the `openet` env (`envs/openet.yml`), for `ee`/`geemap`.

In [9]:
import ee
import geemap

In [2]:
ee.Authenticate()


Successfully saved authorization token.


In [3]:
ee.Initialize(project='focal-pager-510716-m4')

*** Earth Engine *** Share your feedback by taking our Annual Developer Satisfaction Survey: https://google.qualtrics.com/jfe/form/SV_9oS0DRcPvElRMNw?source=python


In [4]:
COL_ID = "projects/openet/assets/disalexi/conus/gridmet/monthly/v2_0"
col = ee.ImageCollection(COL_ID)

In [5]:
print('n images:', col.size().getInfo())
tiles = col.aggregate_array('mgrs_tile').distinct()
print('n MGRS tiles:', tiles.size().getInfo())

n images: 7200
n MGRS tiles: 25


In [6]:
one_per_tile = col.distinct('mgrs_tile')            # first image per tile
footprint = one_per_tile.geometry(1000).dissolve(1000)   # maxError = 1 km
bbox = footprint.bounds(1000).getInfo()['coordinates']
print('bbox (lon, lat):', bbox)

bbox (lon, lat): [[[-125.76945502426096, 25.286086763680956], [-83.52780059175801, 25.286086763680956], [-83.52780059175801, 49.65245313213631], [-125.76945502426096, 49.65245313213631], [-125.76945502426096, 25.286086763680956]]]


In [7]:
def tiles_per_year(y):
    y = ee.Number(y)
    sub = col.filter(ee.Filter.calendarRange(y, y, 'year'))
    return ee.Feature(None, {'year': y,
                             'n_tiles': sub.aggregate_array('mgrs_tile').distinct().size()})

fc = ee.FeatureCollection(ee.List.sequence(2001, 2024).map(tiles_per_year))
print(list(zip(fc.aggregate_array('year').getInfo(),
               fc.aggregate_array('n_tiles').getInfo())))

[(2001, 25), (2002, 25), (2003, 25), (2004, 25), (2005, 25), (2006, 25), (2007, 25), (2008, 25), (2009, 25), (2010, 25), (2011, 25), (2012, 25), (2013, 25), (2014, 25), (2015, 25), (2016, 25), (2017, 25), (2018, 25), (2019, 25), (2020, 25), (2021, 25), (2022, 25), (2023, 25), (2024, 25)]


In [ ]:
footprints = col.distinct('mgrs_tile').geometry()

Map = geemap.Map(center=(38, -100), zoom=4)
Map.addLayer(footprints, {}, 'footprints')
Map  # render the map

Map(center=[38, -100], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', tr…